# Problem 1: Implement a General Gradient Descent Algorithm
---
## Q1: Warm up: construct a generic gradient descent function

In [ ]:
def gd_function(obj, grad, init_guess, step_size, epsilon, max_iter=100):
    """
    Gradient Descent Function to minimize an objective function.

    Parameters:
    - obj: Objective function to minimize (must take a vector
      as input and return a scalar).
    - grad: Gradient function of the objective function
      (returns a vector of gradients).
    - init_guess: Initial guess for the parameters (a vector).
    - step_size: The step size (learning rate) for the gradient descent updates.
    - epsilon: The stopping criterion (the algorithm stops
      when the difference in objective function value on two
      successive steps is below a threshold).
    - max_iter: Maximum number of iterations to prevent infinite loops.

    Returns:
    - opt_point: The vector of parameters that minimize the objective function.
    """
    opt_point = init_guess

    for i in range(max_iter):
        opt_guess = opt_point - step_size * grad(opt_point) # record new guess for the parameters based on last iteration
        if abs(obj(opt_guess)-obj(opt_point)) < epsilon:    # compare threshold
            return opt_guess
        opt_point = opt_guess                               # update the latest guess and continue for loop
    return opt_point

---
## Q2: Use GD function from Q1 to optimize a Gaussian function

### Q2.1 negative Gaussian function

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# A negative Gaussian function
# --------------------------------1. negative Gaussian function 函数定义 ------------------------------
n = 2
u = np.array([10,10])
sigma = np.array([[1000.0, 0.0], 
                  [0.0, 1000.0]])
det_sigma = np.linalg.det(sigma) # sigma 行列式计算
inv_sigma = np.linalg.inv(sigma) # sigma 矩阵求拟

coef = 1.0 / np.sqrt(((2 * np.pi) ** n) * det_sigma) # 分母常数

def neg_gauss(x):
    """
    input :
    x (vector) : data
    
    output:
    fx (vector): negative Gaussian function value of x
    """
    diff = x-u
    exponent = -0.5 * np.dot(diff.T, np.dot(inv_sigma,diff))
    fx = -coef * np.exp(exponent)
    return fx

def neg_gauss_derivative(x):
    """
    input :
    x (vector) : data
    
    output:
    dfdx (vector): derivative of negative Gaussian function
    """
    diff = x-u
    fx = neg_gauss(x)
    dfdx = -fx * np.dot(inv_sigma,diff)
    return dfdx

# —————————————————— 2. 简单的取点和求解梯度函数验证：Test GD function from Q1 ——————————————————

x_gauss = np.array([25.0, 25.0])
step_size_gauss = 10000
epsilon_gauss = 1.0e-15
max_iter_gauess = 10000

opt_point_gauss = gd_function(obj=neg_gauss, grad=neg_gauss_derivative, init_guess=x_gauss
                              , step_size=step_size_gauss, epsilon=epsilon_gauss, max_iter=max_iter_gauess)

print(f"The optimal point of neg_gauss_func (\n"
      f"    Initial_value = {x_gauss},\n"
      f"    step_size = {step_size_gauss},\n"
      f"    epsilon = {epsilon_gauss},\n"
      f"    Max_iter = {max_iter_gauess}\n"
      f") is {opt_point_gauss}")

# ------------------------------- 3. 绘图讨论：函数准备 --------------------------------

# 3.1. 由于需要历次迭代的（梯度范数）数据，因此需要重新定义gd_function函数，使其 Record & Return 历史数据

def gd_function_history(obj, grad, init_guess, step_size, epsilon, max_iter=100):
    """
    Gradient Descent Function to minimize an objective function.

    Parameters:
    - obj: Objective function to minimize (must take a vector
      as input and return a scalar).
    - grad: Gradient function of the objective function
      (returns a vector of gradients).
    - init_guess: Initial guess for the parameters (a vector).
    - step_size: The step size (learning rate) for the gradient descent updates.
    - epsilon: The stopping criterion (the algorithm stops
      when the difference in objective function value on two
      successive steps is below a threshold).
    - max_iter: Maximum number of iterations to prevent infinite loops.

    Returns:
    - opt_point: The vector of parameters that minimize the objective function.
    - point_history: vector 记录走过的每一个坐标点
    - grad_norm: vector 记录每一步梯度的长度（范数)
    """
    opt_point = init_guess
    point_history = [opt_point] # 记录每一个坐标点
    grad_norm = []  # 记录每一个梯度范数
    for i in range(max_iter):
        current_grad = grad(opt_point)
        grad_norm.append(np.linalg.norm(current_grad)) # 记录每一次迭代的向量范数
        
        opt_guess = opt_point - step_size * current_grad # record new guess for the parameters based on last iteration

        if abs(obj(opt_guess)-obj(opt_point)) < epsilon:    # compare threshold
            point_history.append(opt_guess)
            return opt_guess, np.array(point_history), np.array(grad_norm)
        opt_point = opt_guess                               # update the latest guess and continue for loop
        point_history.append(opt_guess)
    return opt_point, np.array(point_history), np.array(grad_norm)

# 3.2. 定义 在作业条件下的 negative gauss function 3D曲线图，对函数的整体形状进行概览，方便后续的梯度优化超参数优化

 # 绘图，由于 negative Gaussian function & quadratic bowl function 都需要绘图，
 # 因此设计一个通用的3D绘图函数，避免代码重复，下同。
def plot_3d_history(obj, point_history, final_point, x_bounds=(-80, 100), y_bounds=(-80, 100), title="3D Surface"):
    """
    通用 3D 曲面与梯度下降轨迹绘图函数
    input:
    - obj: 目标函数
    - point_history: 梯度下降走过的轨迹点集合
    - final_point: 算法收敛的最终点
    - x_bounds: x1 轴的可视化范围 (min, max)
    - y_bounds: x2 轴的可视化范围 (min, max)
    - title: 图表标题
    output: 3D plot of obj function 
    """
    fig = plt.figure(figsize = [10,8])
    ax = fig.add_subplot(111,projection='3d')
    x_range = np.linspace(x_bounds[0],x_bounds[1],100)
    y_range = np.linspace(y_bounds[0],y_bounds[1],100)
    X,Y = np.meshgrid(x_range,y_range)
    Z = np.zeros_like(X)

    # 计算网格点的函数值 (补充逻辑)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i,j] = obj(np.array([X[i,j], Y[i,j]]))

    ax.plot_surface(X,Y,Z,cmap = 'viridis', alpha = 0.6, edgecolor='none')

    # 计算z轴，同时绘制梯度下降的曲线（红色）
    z_history = [obj(pt) for pt in point_history]
    
    ax.plot(point_history[:,0], point_history[:,1], z_history, color = 'red', marker='.', linestyle='-', linewidth = 2, label = 'GD history' )
    
    # 添加起点和终点
    ax.scatter(point_history[0,0],point_history[0,1],z_history[0],color='blue',s=50,label = 'start',zorder = 5)
    ax.scatter(final_point[0],final_point[1],obj(final_point),color='gold',s=150, marker='*', label='Optimal Point', zorder=5)

    ax.set_title(title)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.set_zlabel('f(x)')
    ax.legend() # 修正括号

    plt.tight_layout()
    plt.show()

# 3.3 定义 通用梯度范数演变绘图函数
def plot_grad_norms_history(grad_norms, title="Gradient Norm Evolution"):
    """
    通用梯度范数演变绘图函数
    input:
    - grad_norms: 梯度下降过程中每一步的梯度长度集合
    - title: 图表标题
    
    output: 2D plot of gradient norms evolution 
    """
    fig = plt.figure(figsize=[10, 8])
    ax = fig.add_subplot(111)
    
    ax.plot(grad_norms, color='blue', marker='o', linestyle='-', markersize=3, label='||g||')
    
    ax.set_title(title)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Norm of Gradient (||g||)')
    ax.grid(True)
    ax.legend()
    
    plt.tight_layout()
    plt.show()

# 3.4 定义 通用2D 等高线与梯度下降轨迹绘图函数
def plot_2d_contour_history(obj, point_history, final_point, x_bounds=(-80, 100), y_bounds=(-80, 100), title="2D Contour Plot"):
    """
    通用 2D 等高线与梯度下降轨迹绘图函数
    input:
    - obj: 目标函数
    - point_history: 梯度下降走过的轨迹点集合
    - final_point: 算法收敛的最终点
    - x_bounds: x1 轴的可视化范围 (min, max)
    - y_bounds: x2 轴的可视化范围 (min, max)
    - title: 图表标题
    
    output: 2D contour plot of obj function with GD trajectory
    """
    fig = plt.figure(figsize=[10, 8])
    ax = fig.add_subplot(111)
    
    x_range = np.linspace(x_bounds[0], x_bounds[1], 100)
    y_range = np.linspace(y_bounds[0], y_bounds[1], 100)
    X, Y = np.meshgrid(x_range, y_range)
    Z = np.zeros_like(X)

    # 计算网格点的函数值 
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = obj(np.array([X[i, j], Y[i, j]]))

    # 绘制地形等高线
    contour = ax.contour(X, Y, Z, levels=30, cmap='viridis')
    ax.clabel(contour, inline=True, fontsize=8)

    # 绘制梯度下降的轨迹曲线（红色）
    ax.plot(point_history[:, 0], point_history[:, 1], color='red', marker='x', linestyle='-', linewidth=2, label='GD history')
    
    # 添加起点和终点
    ax.scatter(point_history[0, 0], point_history[0, 1], color='blue', s=50, label='start', zorder=5)
    ax.scatter(final_point[0], final_point[1], color='gold', s=150, marker='*', label='Optimal Point', zorder=5)

    ax.set_title(title)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.legend()

    plt.tight_layout()
    plt.show()


# ------------------------------- 4. 绘图讨论：绘图 --------------------------------

 # 4.1 重新设计合适的梯度下降参数 
x_gauss = np.array([25.0, 25.0])
step_size_gauss = 200000.0 
epsilon_gauss = 1.0e-15
max_iter_gauess = 10000

 # 4.2 获取绘图参数
opt_point_gauss, point_history_gauss, grad_norm_gauss = gd_function_history(
    obj=neg_gauss, 
    grad=neg_gauss_derivative, 
    init_guess=x_gauss, 
    step_size=step_size_gauss, 
    epsilon=epsilon_gauss, 
    max_iter=max_iter_gauess
    )

 # 4.3 调用封装好的函数绘制负高斯函数的 3D 图
plot_3d_history(
    obj=neg_gauss, 
    point_history=point_history_gauss, 
    final_point=opt_point_gauss, 
    x_bounds=(-80, 100), 
    y_bounds=(-80, 100), 
    title="3D Surface of Negative Gaussian"
)

# 4.4 绘制梯度范数演变图
plot_grad_norms_history(
    grad_norms=grad_norm_gauss,
    title="Gradient Norm Evolution of Negative Gaussian"
)

# 4.5 绘制 2D 等高线轨迹图
plot_2d_contour_history(
    obj=neg_gauss, 
    point_history=point_history_gauss, 
    final_point=opt_point_gauss, 
    x_bounds=(-80, 100), 
    y_bounds=(-80, 100), 
    title="2D Contour of Negative Gaussian"
)

# ------------------------------- 5. 绘图讨论：讨论 --------------------------------
# 见作业报告


---
### Q2.2 quadratic bowl function

In [ ]:
# A quadratic bowl function

# ------------------------------------1. 定义二次碗状函数 -----------------------------------
A = np.array([[10,5],[5,10]])
b = np.array([400,400])
def quadratic_bowl(x):
    """
    input :
    x (vector) : data
    
    output:
    fx (vector): quadratic bowl function value of x
    """
    fx = 0.5* np.dot(np.dot(x.T,A),x) - np.dot(x.T,b)
    return fx

def quadratic_bowl_derivative(x):
    """
    input :
    x (vector) : data
    
    output:
    dfdx (vector): derivative of quadratic bowl function
    """
    dfdx = np.dot(A,x) - b
    return dfdx

# -----------------------------2. 简单的取点和求解梯度函数验证：Test GD function from Q1 --------------------------

x_bowl = np.array([5.0, 5.0])
step_size_bowl = 0.1
epsilon_bowl = 1.0e-20
max_iter_bowl = 10000

opt_point_bowl = gd_function(obj=quadratic_bowl, 
                             grad=quadratic_bowl_derivative, 
                             init_guess=x_bowl, 
                             step_size=step_size_bowl, 
                             epsilon=epsilon_bowl, 
                             max_iter=max_iter_bowl
                             )

print(f"The optimal point of quadratic_bowl_func (\n"
      f"    Initial_value = {x_bowl},\n"
      f"    step_size = {step_size_bowl},\n"
      f"    epsilon = {epsilon_bowl},\n"
      f"    Max_iter = {max_iter_bowl}\n"
      f") is {opt_point_bowl}")


# ------------------------------- 3. 绘图讨论：绘图 --------------------------------

 # 4.1 重新设计合适的梯度下降参数 
x_bowl = np.array([75.0, 50.0])
step_size_bowl = 0.1
epsilon_bowl = 1.0e-15
max_iter_bowl = 10000

 # 4.2 获取绘图参数
opt_point_bowl, point_history_bowl, grad_norm_bowl = gd_function_history(
                             obj=quadratic_bowl, 
                             grad=quadratic_bowl_derivative, 
                             init_guess=x_bowl, 
                             step_size=step_size_bowl, 
                             epsilon=epsilon_bowl, 
                             max_iter=max_iter_bowl
                             )

 # 4.3 调用封装好的函数绘制负高斯函数的 3D 图
plot_3d_history(
    obj=quadratic_bowl, 
    point_history=point_history_bowl, 
    final_point=opt_point_bowl, 
    x_bounds=(-80, 100), 
    y_bounds=(-80, 100), 
    title="3D Surface of Quadratic Bowl"
)

# 4.4 绘制梯度范数演变图
plot_grad_norms_history(
    grad_norms=grad_norm_bowl,
    title="Gradient Norm Evolution of Quadratic Bowl"
)

# 4.5 绘制 2D 等高线轨迹图
plot_2d_contour_history(
    obj=quadratic_bowl, 
    point_history=point_history_bowl, 
    final_point=opt_point_bowl, 
    x_bounds=(-80, 100), 
    y_bounds=(-80, 100), 
    title="2D Contour of Quadratic Bowl"
)

# ------------------------------- 4. 绘图讨论：讨论 --------------------------------
# 见作业报告

## Q3: Numerical gradient

In [ ]:
import pandas as pd
# Only need to use the negative Gaussian function as the target in Q3 problem.

# -------------------------------1. 实现数值梯度函数----------------------------------

def neg_gauss_derivative_num(x):
    """
    input :
    x (vector) : data
    
    output:
    dfdx (vector): numerical  derivative of negative Gaussian function
    """
    dfdx = np.zeros_like(x)
    epsilon_num = 1.0e-5
    for i in range(len(x)):
        ei = np.zeros_like(x)
        ei[i] = 1.0
        dfdx[i] = (neg_gauss(x+epsilon_num*ei)-neg_gauss(x-epsilon_num*ei))/(2*epsilon_num)

    return dfdx


#---------------------------------2. 使用梯度数值进行梯度下降-------------------------------------
 # 4.1 重新设计合适的梯度下降参数 
x_gauss_num = np.array([50.0, 50.0])
step_size_gauss_num = 200000.0 
epsilon_gauss_num = 1.0e-15
max_iter_gauess_num = 10000

opt_point_gauss_num = gd_function(obj=neg_gauss, grad=neg_gauss_derivative_num, init_guess=x_gauss_num
                              , step_size=step_size_gauss_num, epsilon=epsilon_gauss_num, max_iter=max_iter_gauess_num)

print(f"The optimal point of neg_gauss_func (\n"
      f"    Initial_value = {x_gauss_num},\n"
      f"    step_size = {step_size_gauss_num},\n"
      f"    epsilon = {epsilon_gauss_num},\n"
      f"    Max_iter = {max_iter_gauess_num}\n"
      f") is {opt_point_gauss_num}")


 # 4.2 获取绘图参数
opt_point_gauss_num, point_history_gauss_num, grad_norm_gauss_num = gd_function_history(
    obj=neg_gauss, 
    grad=neg_gauss_derivative_num, 
    init_guess=x_gauss_num, 
    step_size=step_size_gauss_num, 
    epsilon=epsilon_gauss_num, 
    max_iter=max_iter_gauess_num
    )

 # 4.3 调用封装好的函数绘制负高斯函数的 3D 图
plot_3d_history(
    obj=neg_gauss, 
    point_history=point_history_gauss_num, 
    final_point=opt_point_gauss_num, 
    x_bounds=(-80, 100), 
    y_bounds=(-80, 100), 
    title="3D Surface of Numerical Negative Gaussian"
)

# 4.4 绘制梯度范数演变图
plot_grad_norms_history(
    grad_norms=grad_norm_gauss_num,
    title="Gradient Norm Evolution of  Numerical Negative Gaussian"
)

# 4.5 绘制 2D 等高线轨迹图
plot_2d_contour_history(
    obj=neg_gauss, 
    point_history=point_history_gauss_num, 
    final_point=opt_point_gauss_num, 
    x_bounds=(-80, 100), 
    y_bounds=(-80, 100), 
    title="2D Contour of Numerical Negative Gaussian"
)



### 附1：比较数值梯度和解析梯度

In [ ]:
# ----------------- 1. 创建全域网格点并计算梯度 -----------------
x_vals = np.linspace(-40, 60, 20)
y_vals = np.linspace(-40, 60, 20)
X, Y = np.meshgrid(x_vals, y_vals)

# 初始化储存矩阵
U_ana = np.zeros_like(X) # 解析梯度 dx1
V_ana = np.zeros_like(Y) # 解析梯度 dx2
U_num = np.zeros_like(X) # 数值梯度 dx1
V_num = np.zeros_like(Y) # 数值梯度 dx2
Error = np.zeros_like(X) # L2 绝对误差

for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        pt = np.array([X[i,j], Y[i,j]])
        
        grad_a = neg_gauss_derivative(pt)
        grad_n = neg_gauss_derivative_num(pt)
        
        U_ana[i,j], V_ana[i,j] = grad_a[0], grad_a[1]
        U_num[i,j], V_num[i,j] = grad_n[0], grad_n[1]
        Error[i,j] = np.linalg.norm(grad_a - grad_n)

# ----------------- 图 1：向量场重叠  -----------------
fig1, ax1 = plt.subplots(figsize=(7, 6))

# 蓝色粗箭头代表解析解，红色细箭头代表数值解
ax1.quiver(X, Y, U_ana, V_ana, color='blue', alpha=0.6, label='Analytical', width=0.005)
ax1.quiver(X, Y, U_num, V_num, color='red', alpha=0.6, label='Numerical', width=0.002)

ax1.set_title("Vector Field Overlay\n(Blue = Analytical, Red = Numerical)", fontsize=14)
ax1.set_xlabel("x1", fontsize=12)
ax1.set_ylabel("x2", fontsize=12)
ax1.legend(loc='upper right')

fig1.tight_layout()
fig1.savefig('gradient_vector_field.png', dpi=300, bbox_inches='tight')
plt.show(fig1)

# ----------------- 图 2：L2 误差热力图 ---------------------

fig2, ax2 = plt.subplots(figsize=(7, 6))

c = ax2.contourf(X, Y, Error, levels=20, cmap='inferno')
# 强制使用科学计数法显示误差尺度，体现极其微小的误差
cbar = fig2.colorbar(c, ax=ax2, format='%.1e') 
cbar.set_label('L2 Error Magnitude', fontsize=12)

ax2.set_title("L2 Error Heatmap\n|| Analytical - Numerical ||", fontsize=14)
ax2.set_xlabel("x1", fontsize=12)
ax2.set_ylabel("x2", fontsize=12)

fig2.tight_layout()
fig2.savefig('gradient_error_heatmap.png', dpi=300, bbox_inches='tight')
plt.show(fig2)

# ----------------- 图 3：一维横截面梯度对比  --------------

# 固定 x2 = 10，看 x1 变化时梯度的贴合度
x_line = np.linspace(-40, 60, 100)
grad_x_ana = []
grad_x_num = []

for x in x_line:
    pt = np.array([x, 10.0])
    grad_x_ana.append(neg_gauss_derivative(pt)[0])
    grad_x_num.append(neg_gauss_derivative_num(pt)[0])

fig3, ax3 = plt.subplots(figsize=(7, 6))

ax3.plot(x_line, grad_x_ana, 'b-', label='Analytical dx1', linewidth=4, alpha=0.4)
ax3.plot(x_line, grad_x_num, 'r--', label='Numerical dx1', linewidth=2)

ax3.set_title("Gradient Component Cross-section\n(Fixed at x2 = 10)", fontsize=14)
ax3.set_xlabel("x1", fontsize=12)
ax3.set_ylabel("Gradient magnitude (dx1)", fontsize=12)
ax3.legend(loc='upper right')
ax3.grid(True, linestyle='--', alpha=0.6)

fig3.tight_layout()
fig3.savefig('gradient_cross_section.png', dpi=300, bbox_inches='tight')
plt.show(fig3)

### 附2：负高斯函数，调参绘图

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --------------------------------1. Negative Gaussian Function ------------------------------
n = 2
u = np.array([10.0, 10.0])
sigma = np.array([[1000.0, 0.0], 
                  [0.0, 1000.0]])
det_sigma = np.linalg.det(sigma)
inv_sigma = np.linalg.inv(sigma)
coef = 1.0 / np.sqrt(((2 * np.pi) ** n) * det_sigma)

def neg_gauss(x):
    diff = x - u
    exponent = -0.5 * np.dot(diff.T, np.dot(inv_sigma, diff))
    fx = -coef * np.exp(exponent)
    return fx

def neg_gauss_derivative(x):
    diff = x - u
    fx = neg_gauss(x)
    dfdx = -fx * np.dot(inv_sigma, diff)
    return dfdx

# --------------------------------2. 记录历史轨迹的 GD Function ------------------------------
def gd_function_history(obj, grad, init_guess, step_size, epsilon, max_iter=1000):
    opt_point = np.array(init_guess, dtype=float)
    point_history = [opt_point.copy()]
    grad_norm = []
    
    for i in range(max_iter):
        current_grad = grad(opt_point)
        grad_norm.append(np.linalg.norm(current_grad))
        
        opt_guess = opt_point - step_size * current_grad
        
        if abs(obj(opt_guess) - obj(opt_point)) < epsilon:
            point_history.append(opt_guess.copy())
            return opt_guess, np.array(point_history), np.array(grad_norm)
            
        opt_point = opt_guess
        point_history.append(opt_guess.copy())
        
    return opt_point, np.array(point_history), np.array(grad_norm)

# --------------------------------3. 四组对照实验参数设定 ------------------------------
# 分别对应：正常收敛、学习率过大(震荡)、学习率过小(收敛慢)、初始值过大且阈值过大(过早停止)
configs = [
    {"name": "1. Normal (Converges)", 
     "x0": [25.0, 25.0], "lr": 20000.0, "eps": 1e-15, "max_iter": 5000},
     
    {"name": "2. LR Too Large (Oscillates/Overshoots)", 
     "x0": [25.0, 25.0], "lr": 9500000.0, "eps": 1e-15, "max_iter": 5000},
     
    {"name": "3. LR Too Small (Slow Convergence)", 
     "x0": [25.0, 25.0], "lr": 100.0,  "eps": 1e-15, "max_iter": 5000},
     
    {"name": "4. Far Init", 
     "x0": [80.0, 80.0], "lr": 20000.0, "eps": 1e-15, "max_iter": 5000},

    {"name": "5. Large Eps (Premature Stop)", 
     "x0": [25.0, 25.0], "lr": 20000.0, "eps": 1e-9, "max_iter": 5000}
]

# --------------------------------4. 计算全局绘图网格参数 ------------------------------
# 为了让四幅图的地形等高线保持一致，在这里统一计算 Z
x_bounds = (-20, 100)
y_bounds = (-20, 100)
x_range = np.linspace(x_bounds[0], x_bounds[1], 100)
y_range = np.linspace(y_bounds[0], y_bounds[1], 100)
X, Y = np.meshgrid(x_range, y_range)
Z = np.zeros_like(X)

for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i,j] = neg_gauss(np.array([X[i,j], Y[i,j]]))

# --------------------------------5. 绘制 4x3 的总图表 ------------------------------
# 设定一张大图表 figsize=(宽度, 高度)
fig = plt.figure(figsize=(20, 24))

for i, cfg in enumerate(configs):
    # 1. 运行梯度下降
    opt_point, point_history, grad_norm = gd_function_history(
        obj=neg_gauss, grad=neg_gauss_derivative, 
        init_guess=cfg["x0"], step_size=cfg["lr"], 
        epsilon=cfg["eps"], max_iter=cfg["max_iter"]
    )
    
    # ------------------ 第1列：3D曲面图 ------------------
    ax1 = fig.add_subplot(5, 3, i*3 + 1, projection='3d')
    ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.6, edgecolor='none')
    z_history = [neg_gauss(pt) for pt in point_history]
    
    ax1.plot(point_history[:,0], point_history[:,1], z_history, color='red', marker='.', linestyle='-', linewidth=2)
    ax1.scatter(point_history[0,0], point_history[0,1], z_history[0], color='blue', s=80, label='Start', zorder=5)
    ax1.scatter(opt_point[0], opt_point[1], neg_gauss(opt_point), color='gold', s=200, marker='*', label='End', zorder=5)
    
    ax1.set_title(f"[{cfg['name']}]\n3D Surface")
    ax1.set_xlabel('x1')
    ax1.set_ylabel('x2')
    ax1.set_zlabel('f(x)')
    
    # ------------------ 第2列：梯度范数变化图 ------------------
    ax2 = fig.add_subplot(5, 3, i*3 + 2)
    ax2.plot(grad_norm, color='blue', marker='o', linestyle='-', markersize=4)
    ax2.set_title(f"Grad Norm Evolution\nlr={cfg['lr']}, eps={cfg['eps']}")
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('||g|| (Norm of Gradient)')
    ax2.grid(True, linestyle='--', alpha=0.6)
    
    # ------------------ 第3列：2D等高线轨迹图 ------------------
    ax3 = fig.add_subplot(5, 3, i*3 + 3)
    contour = ax3.contour(X, Y, Z, levels=30, cmap='viridis')
    ax3.plot(point_history[:,0], point_history[:,1], color='red', marker='x', linestyle='-', linewidth=2)
    ax3.scatter(point_history[0,0], point_history[0,1], color='blue', s=80, zorder=5, label='Start')
    ax3.scatter(opt_point[0], opt_point[1], color='gold', s=200, marker='*', zorder=5, label='End')
    
    ax3.set_title(f"2D Contour\nStart: {cfg['x0']}")
    ax3.set_xlabel('x1')
    ax3.set_ylabel('x2')
    
    # 只有第一行需要加图例，避免重复杂乱
    if i == 0:
        ax3.legend()

# 自动调整子图间距，防止字尊重叠
plt.tight_layout(pad=3.0)

# 保存图片（dpi=300保证报告贴图极其清晰）
plt.savefig('gd_hyperparameter_analysis.png', dpi=300, bbox_inches='tight')

# 显示图片
plt.show()

### 二次碗函数，调参绘图

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --------------------------------1. Quadratic Bowl Function ------------------------------
A = np.array([[10, 5], [5, 10]])
b = np.array([400, 400])

def quadratic_bowl(x):
    fx = 0.5 * np.dot(np.dot(x.T, A), x) - np.dot(x.T, b)
    return fx

def quadratic_bowl_derivative(x):
    dfdx = np.dot(A, x) - b
    return dfdx

# --------------------------------2. 记录历史轨迹的 GD Function ------------------------------
def gd_function_history(obj, grad, init_guess, step_size, epsilon, max_iter=1000):
    opt_point = np.array(init_guess, dtype=float)
    point_history = [opt_point.copy()]
    grad_norm = []
    
    for i in range(max_iter):
        current_grad = grad(opt_point)
        grad_norm.append(np.linalg.norm(current_grad))
        
        opt_guess = opt_point - step_size * current_grad
        
        # 增加一个防止发散报错的保护机制
        if np.linalg.norm(opt_guess) > 1e10:
            print("Warning: Divergence detected, stopping early.")
            break
            
        if abs(obj(opt_guess) - obj(opt_point)) < epsilon:
            point_history.append(opt_guess.copy())
            return opt_guess, np.array(point_history), np.array(grad_norm)
            
        opt_point = opt_guess
        point_history.append(opt_guess.copy())
        
    return opt_point, np.array(point_history), np.array(grad_norm)

# --------------------------------3. 五组对照实验参数设定 ------------------------------
# 分别对应：正常收敛、学习率过大(震荡)、学习率过小(收敛慢)、初始点极远、容差极大(早停)
configs = [
    {"name": "1. Normal (Converges)", 
     "x0": [75.0, 50.0], "lr": 0.05, "eps": 1e-15, "max_iter": 500},
     
    {"name": "2. LR Too Large (Oscillates/Overshoots)", 
     # 二次函数的最大特征值为 15，所以理论上步长 > 2/15 (~0.133) 就会开始不稳定，0.125会导致剧烈震荡
     "x0": [75.0, 50.0], "lr": 0.125, "eps": 1e-15, "max_iter": 500}, 
     
    {"name": "3. LR Too Small (Slow Convergence)", 
     "x0": [75.0, 50.0], "lr": 0.005,  "eps": 1e-15, "max_iter": 500},
     
    {"name": "4. Far Init Point (Normal LR)", 
     "x0": [350.0, -250.0], "lr": 0.05, "eps": 1e-15, "max_iter": 500},
     
    {"name": "5. Large Eps (Premature Stop)", 
     # 故意设一个非常大的阈值，让它在刚开始迭代时就误以为到达谷底
     "x0": [75.0, 50.0], "lr": 0.05, "eps": 300, "max_iter": 500} 
]

# --------------------------------4. 计算全局绘图网格参数 ------------------------------
# 为了让所有图视角一致，根据最远初始点扩展绘图边界
x_bounds = (-50, 400)
y_bounds = (-300, 150)
x_range = np.linspace(x_bounds[0], x_bounds[1], 100)
y_range = np.linspace(y_bounds[0], y_bounds[1], 100)
X, Y = np.meshgrid(x_range, y_range)
Z = np.zeros_like(X)

for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i,j] = quadratic_bowl(np.array([X[i,j], Y[i,j]]))

# --------------------------------5. 绘制 5x3 的总图表 ------------------------------
fig = plt.figure(figsize=(20, 30))

for i, cfg in enumerate(configs):
    # 1. 运行梯度下降
    opt_point, point_history, grad_norm = gd_function_history(
        obj=quadratic_bowl, grad=quadratic_bowl_derivative, 
        init_guess=cfg["x0"], step_size=cfg["lr"], 
        epsilon=cfg["eps"], max_iter=cfg["max_iter"]
    )
    
    # ------------------ 第1列：3D曲面图 ------------------
    ax1 = fig.add_subplot(5, 3, i*3 + 1, projection='3d')
    ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.6, edgecolor='none')
    z_history = [quadratic_bowl(pt) for pt in point_history]
    
    ax1.plot(point_history[:,0], point_history[:,1], z_history, color='red', marker='.', linestyle='-', linewidth=2)
    ax1.scatter(point_history[0,0], point_history[0,1], z_history[0], color='blue', s=80, label='Start', zorder=5)
    ax1.scatter(opt_point[0], opt_point[1], quadratic_bowl(opt_point), color='gold', s=200, marker='*', label='End', zorder=5)
    
    ax1.set_title(f"[{cfg['name']}]\n3D Surface")
    ax1.set_xlabel('x1')
    ax1.set_ylabel('x2')
    ax1.set_zlabel('f(x)')
    
    # ------------------ 第2列：梯度范数变化图 ------------------
    ax2 = fig.add_subplot(5, 3, i*3 + 2)
    ax2.plot(grad_norm, color='blue', marker='o', linestyle='-', markersize=4)
    ax2.set_title(f"Grad Norm Evolution\nlr={cfg['lr']}, eps={cfg['eps']}")
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('||g|| (Norm of Gradient)')
    ax2.grid(True, linestyle='--', alpha=0.6)
    
    # 对于震荡现象，采用对数坐标更能看清趋势
    if "Large" in cfg["name"] and "LR" in cfg["name"]:
        ax2.set_yscale('log')
    
    # ------------------ 第3列：2D等高线轨迹图 ------------------
    ax3 = fig.add_subplot(5, 3, i*3 + 3)
    contour = ax3.contour(X, Y, Z, levels=40, cmap='viridis')
    ax3.plot(point_history[:,0], point_history[:,1], color='red', marker='x', linestyle='-', linewidth=2)
    ax3.scatter(point_history[0,0], point_history[0,1], color='blue', s=80, zorder=5, label='Start')
    ax3.scatter(opt_point[0], opt_point[1], color='gold', s=200, marker='*', zorder=5, label='End')
    
    ax3.set_title(f"2D Contour\nStart: {cfg['x0']}")
    ax3.set_xlabel('x1')
    ax3.set_ylabel('x2')
    
    # 只有第一行需要加图例，避免重复杂乱
    if i == 0:
        ax3.legend()

# 自动调整子图间距，防止字尊重叠
plt.tight_layout(pad=3.0)

# 保存极清图片
plt.savefig('gd_quadratic_bowl_analysis.png', dpi=300, bbox_inches='tight')

# 显示图片
plt.show()